# Resultados (dev CV), análise de erros e interpretação

Este notebook usa **apenas as previsões out-of-fold do conjunto dev**. O conjunto de teste fica
fechado até à avaliação final (`python -m murmur evaluate`). Antes de o correr é preciso ter as
experiências treinadas (`python -m murmur cv ...`).

`RESULTS_DIR` (default `outputs`) escolhe a pasta das experiências; `MAIN` as duas a destacar.

In [ ]:
import os
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # the experiment configs use paths relative to the project root
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from murmur.data import CLASSES, load_signals
from murmur.evaluation import crossfit_decision, patient_scores
from murmur.interpret import attribution_by_state, load_segmentation, state_per_frame, time_profile
from murmur.plots import SERIES, plot_confusion_matrix, plot_cv_comparison, plot_learning_curves, setup_style
from murmur.pipeline import describe, fold_models, load_dataset, load_experiment

%matplotlib inline
setup_style()
OUT = Path(os.environ.get("RESULTS_DIR", "outputs"))
EXPS = sorted(p.parent.name for p in OUT.glob("*/cv_summary.json") if not p.parent.name.endswith("_all"))
MAIN = [e for e in ("cnn", "rnn") if e in EXPS] or EXPS[:2]
print("experiments:", EXPS, "| main:", MAIN)

## 1. Comparação das experiências (WA por paciente, média ± desvio-padrão nos 5 folds)

In [ ]:
rows = []
for e in EXPS:
    cfg, s, _ = load_experiment(OUT / e)
    d = s["decision"]
    model, feats = describe(cfg)
    row = {"experiment": e, "model": model, "features": feats, "decision": f"{d['method']} + {'tuned' if d['tuned'] else 'argmax'}",
           "WA mean": d["wa_mean"], "WA std": d["wa_std"]}
    for m, r in s["methods"].items():
        row[f"{m}/argmax"] = r["argmax"]["wa_mean"]
        row[f"{m}/tuned"] = r["tuned"]["wa_mean"]
    rows.append(row)
table = pd.DataFrame(rows).sort_values("WA mean", ascending=False).round(3)
display(table)
fig = plot_cv_comparison(table["experiment"].to_numpy(), table["WA mean"].to_numpy(), table["WA std"].to_numpy())
plt.show()

**Como ler.** As colunas `mean/…` e `max/…` comparam as duas agregações gravação → paciente, e
`argmax` vs `tuned` diz se afinar os multiplicadores ajudou em folds não vistos (cross-fitting).
Diferenças menores do que o desvio-padrão entre folds (≈ 0.02–0.04) **não** são significativas.

## 2. Curvas de aprendizagem

In [ ]:
for e in MAIN:
    cfg, _, _ = load_experiment(OUT / e)
    histories = [pd.read_csv(p) for p in sorted((OUT / e).glob("fold*/history.csv"))]
    model, feats = describe(cfg)
    plot_learning_curves(histories, f"{e}: {model}, {feats}")
    plt.show()

**Como ler.** Se a perda de treino continua a descer e a de validação sobe, há sobreajuste (mais
regularização / *augmentation*, ou um modelo mais pequeno). Se as duas ficam altas, há subajuste.
A WA de validação é ruidosa (~150 pacientes por fold, poucos Unknown), e por isso a época
escolhida tem alguma variância.

## 3. Matrizes de confusão (out-of-fold, conjunto dev)

In [ ]:
fig, axes = plt.subplots(1, len(MAIN), figsize=(4.4 * len(MAIN), 3.8))
for ax, e in zip(np.atleast_1d(axes), MAIN):
    d = load_experiment(OUT / e)[1]["decision"]
    plot_confusion_matrix(d["confusion_matrix"], f"{e} (WA = {d['pooled_wa']:.3f})", ax=ax)
fig.tight_layout()
plt.show()

## 4. Análise de erros: que pacientes falham?

In [ ]:
cfg0 = load_experiment(OUT / MAIN[0])[0]
patients, recordings, patient_labels, splits = load_dataset(cfg0)

def oof_patient_predictions(exp):
    cfg, s, oof = load_experiment(OUT / exp)
    d = s["decision"]
    scores = patient_scores(oof, d["method"])
    y = patient_labels.loc[scores.index].to_numpy()
    folds = oof.groupby("patient_id")["fold"].first().loc[scores.index].to_numpy()
    pred = crossfit_decision(scores.to_numpy(), y, folds)["pred"]["tuned" if d["tuned"] else "argmax"]
    return pd.DataFrame({"patient_id": scores.index, "true": [CLASSES[i] for i in y], "pred": [CLASSES[i] for i in pred]})

errors = patients.copy()
for e in MAIN:
    errors = errors.merge(oof_patient_predictions(e).rename(columns={"pred": f"pred_{e}"}).drop(columns="true"), on="patient_id")
present = errors[errors["murmur"] == "Present"]
for col in ["systolic_murmur_grading", "systolic_murmur_pitch", "systolic_murmur_timing", "most_audible_location"]:
    t = present.groupby(col)[[f"pred_{e}" for e in MAIN]].agg(lambda p: (p == "Present").mean()).round(2)
    t.insert(0, "n", present.groupby(col).size())
    print(f"recall of Present by {col}:")
    display(t)

In [ ]:
for col in ["campaign", "age"]:
    t = errors.groupby(col)[[f"pred_{e}" for e in MAIN]].agg(lambda p: (p == errors.loc[p.index, "murmur"]).mean()).round(2)
    t.insert(0, "n", errors.groupby(col).size())
    print(f"accuracy by {col}:")
    display(t)
print("what are the Unknown patients predicted as?")
display(errors.loc[errors["murmur"] == "Unknown", [f"pred_{e}" for e in MAIN]].apply(pd.Series.value_counts).fillna(0).astype(int))

**Interpretar e agir.** Se o recall de Present cair muito nos sopros I/VI ou de *pitch* baixo, o
modelo falha os sopros subtis. Ideias: janelas mais longas, mais resolução nas baixas
frequências (mais bandas mel abaixo de 200 Hz), *augmentation* específica. Se uma campanha for
muito pior, há *domain shift* entre campanhas. Os Unknown são difíceis por definição
(o especialista também não decidiu).

## 5. Onde olham os modelos?

Para gravações de validação com sopro (fold 0, modelo do fold 0) compara-se a parte da atenção
(RNN) ou da saliência |gradiente × entrada| (CNN) que cai em cada fase do ciclo cardíaco com a
fração de tempo dessa fase. Se `weight_share` na sístole for bem maior do que `time_share`, o
modelo olha para onde estão os sopros. Isto é evidência de que aprendeu o padrão certo e não
um atalho (*shortcut*).

In [ ]:
device = torch.device("cpu")
for e in MAIN:
    cfg, _, _ = load_experiment(OUT / e)
    kind = cfg["model"]["type"]
    _, recs, _, _ = load_dataset(cfg)
    signals = load_signals(recs, cfg["audio"], cfg.get("cache_dir"))
    fold, model = next(fold_models(OUT / e, device))
    val = recs[(recs["subset"] == "dev") & (recs["fold"] == fold)]
    for label, name in [(0, "Present"), (2, "Absent")]:
        shares, n = attribution_by_state(model, kind, cfg, val[val["label"] == label], signals, max_windows=200)
        if shares is not None:
            print(f"{e} ({'attention' if kind == 'rnn' else 'saliency'}) on {n} windows of {name} recordings:")
            display(shares.round(3))

In [ ]:
def show_profile(exp, rec_id, seconds=4.0):
    cfg, _, _ = load_experiment(OUT / exp)
    _, recs, _, _ = load_dataset(cfg)
    rec = recs.set_index("rec_id").loc[rec_id]
    signals = load_signals(recs, cfg["audio"], cfg.get("cache_dir"))
    fs = cfg["audio"]["fs"]
    x = signals[rec_id].astype(np.float32)[: int(seconds * fs)]
    _, model = next(fold_models(OUT / exp, torch.device("cpu")))
    probs, w = time_profile(model, x, cfg["model"]["type"])
    hop_s = cfg["features"]["hop_length"] / fs
    seg = load_segmentation(rec["wav_path"])
    fig, ax = plt.subplots(figsize=(10, 2.8))
    ax.plot(np.arange(len(x)) / fs, x / np.abs(x).max(), lw=0.5, color=SERIES[0], alpha=0.7, label="signal")
    ax.plot(np.arange(len(w)) * hop_s, w / w.max(), lw=2, color=SERIES[1], label="attention" if cfg["model"]["type"] == "rnn" else "saliency")
    if seg is not None:
        for _, s in seg[(seg["state"] == 2) & (seg["start"] < seconds)].iterrows():
            ax.axvspan(s["start"], min(s["end"], seconds), color=SERIES[2], alpha=0.15, lw=0)
    ax.set_title(f"{exp} on {rec_id} (green = systole): P(Present) = {probs[0]:.2f}")
    ax.set_xlabel("Time (s)")
    ax.legend(loc="upper right")
    plt.show()

cfg0 = load_experiment(OUT / MAIN[0])[0]
_, recs0, _, _ = load_dataset(cfg0)
example = recs0[(recs0["subset"] == "dev") & (recs0["fold"] == 0) & (recs0["label"] == 0)]["rec_id"].iloc[0]
for e in MAIN:
    show_profile(e, example)

## 6. Notas para os slides

Preencher depois de correr as experiências (ver também `experiments.md`):

- melhor configuração de cada modelo e WA de CV (média ± dp);
- o que mudou entre versões e porquê (curvas, matrizes de confusão, análise de erros);
- o que a interpretação mostra (atenção/saliência na sístole?).